In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **Prepare Data**

In [ ]:
!pip -q install torch torchvision transformers timm

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from timm import create_model

In [ ]:
class HouseDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.data.iloc[idx, 0])
        image = Image.open(img_path).convert('RGB')
        label = torch.tensor(self.data.iloc[idx, 1], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),  # Flip images randomly
    transforms.RandomRotation(10),  # Rotate images slightly
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Adjust colors
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
train_dataset = HouseDataset("/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train.csv", "/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/train", transform=transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# **Model**

In [ ]:
import timm

# timm.list_models("")

In [ ]:
# model = create_model("swin_base_patch4_window7_224", pretrained=True, num_classes=1) #0.97
model = create_model("convnextv2_tiny", pretrained=True, num_classes=1) # 0.9888
# model = create_model("convnextv2_base", pretrained=True, num_classes=1)
model = model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
def train_model(model, dataloader, criterion, optimizer, epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {running_loss/len(dataloader):.4f}")

In [ ]:
train_model(model, train_loader, criterion, optimizer, epochs=20)

Epoch 1, Loss: 0.1002
Epoch 2, Loss: 0.0591
Epoch 3, Loss: 0.0390
Epoch 4, Loss: 0.0438
Epoch 5, Loss: 0.0432
Epoch 6, Loss: 0.0335
Epoch 7, Loss: 0.0169
Epoch 8, Loss: 0.0208
Epoch 9, Loss: 0.0229
Epoch 10, Loss: 0.0069
Epoch 11, Loss: 0.0325
Epoch 12, Loss: 0.0114
Epoch 13, Loss: 0.0281
Epoch 14, Loss: 0.0235
Epoch 15, Loss: 0.0076
Epoch 16, Loss: 0.0229
Epoch 17, Loss: 0.0137
Epoch 18, Loss: 0.0140
Epoch 19, Loss: 0.0243
Epoch 20, Loss: 0.0082


In [ ]:
torch.save(model.state_dict(), "convnextv2_base_house_model.pth")

Test folder image preprocessing

# **Prediction**

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [ ]:
import torch
import os
import pandas as pd
from PIL import Image
from torchvision import transforms

def predict(model, test_folder, output_csv):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    test_images = [f for f in os.listdir(test_folder) if f.lower().endswith(('png', 'jpg', 'jpeg'))]  # Ensure image files only
    results = []

    with torch.no_grad():
        for img_name in test_images:
            img_path = os.path.join(test_folder, img_name)
            image = Image.open(img_path).convert('RGB')
            image = test_transform(image).unsqueeze(0).to(device)  # Apply transformation

            output = model(image)
            if isinstance(output, torch.Tensor):
                output = output.item()  # Ensure conversion if tensor

            pred = 1 if output > 0 else 0  # Convert to binary prediction
            results.append([os.path.splitext(img_name)[0], pred])  # Remove extension safely

    df = pd.DataFrame(results, columns=["id", "answer"])
    df.to_csv(output_csv, index=False)
    print(f"Predictions saved to {output_csv}")

In [ ]:
predict(model, "/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test", "submission.csv")

Predictions saved to submission.csv
